# Metadata inference (optional notebook)

The supported runner is the local CLI script `scripts/run_metadata.py` via Google Colab CLI.
See the README for `colab new`, `colab drivemount`, and `colab exec`.

Code is synced onto the VM with git (`scripts/install_package.py` clones into `/content/open-enricher`).
Drive is only for videos. Push commits before you expect the VM to see library changes.

This notebook is a leftover for inspecting a session that already has Drive mounted
(for example after `colab drivemount`). It does not call `drive.mount`.

If you run it with `colab exec -f notebooks/01_metadata_inference.ipynb`, raise `--timeout`
(model load and video inference need thousands of seconds, not the CLI default of 30).

In [ ]:
# Drive is for videos. Library code lives at the git checkout on the VM.
REPO_DIR = "/content/open-enricher"
DATA_ROOT = "/content/drive/MyDrive/open_enricher/data"
OUTPUT_DIR = "/content/drive/MyDrive/open_enricher/outputs"
VIDEO = ""

In [ ]:
import sys
from pathlib import Path

repo = Path(REPO_DIR)
assert repo.exists(), f"Repository not found: {repo}"
req_file = str(repo / "requirements.txt")
repo_str = str(repo)

%pip install -q -r {req_file}
%pip install -q -e {repo_str}

src = str(repo / "src")
if src not in sys.path:
    sys.path.insert(0, src)

In [ ]:
from open_enricher import (
    DriveConfig,
    MetadataConfig,
    ModelConfig,
    extract_metadata,
    list_videos,
    load_model,
    resolve_video,
    save_json,
)

model_config = ModelConfig(model_id="Qwen/Qwen3-VL-2B-Instruct")
drive_config = DriveConfig(root=DATA_ROOT, output_dir=OUTPUT_DIR)
metadata_config = MetadataConfig(num_frames=16)

videos = list_videos(drive_config.root)
print(f"Found {len(videos)} video(s) under {drive_config.root}")
for path in videos[:10]:
    print(path)

video_path = resolve_video(VIDEO, drive_config.root) if VIDEO else videos[0]
print("Using:", video_path)

In [ ]:
model, processor = load_model(model_config)
result = extract_metadata(
    model,
    processor,
    video_path,
    metadata_config,
    model_id=model_config.model_id,
)

out_path = Path(drive_config.output_dir) / "metadata" / f"{video_path.stem}.json"
save_json(out_path, result)
print("Saved:", out_path)
result